In [42]:
import kagglehub
import pandas as pd
import numpy as np
from xgboost import XGBClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.preprocessing import LabelEncoder

In [43]:
# ============== Preparing dataframes ==============
path = kagglehub.competition_download("titanic")
train_df = pd.read_csv(f"{path}/train.csv")
test_df = pd.read_csv(f"{path}/test.csv")

In [47]:
# ============== Processing ==============
def extract_title(name):
    title = name.split(',')[1].split('.')[0].strip()
    # grouping of rare titles
    if title in ['Lady', 'Countess', 'Capt', 'Col', 'Don', 'Dr', 'Major', 'Rev', 'Sir', 'Jonkheer', 'Dona']:
        return 'Rare'
    return title

# Train
train = train_df.copy()
train = train.drop(['Name', 'Ticket', 'PassengerId'], axis=1)
train['Sex'] = train['Sex'].map({'male': 0, 'female': 1})
mean_age = train['Age'].mean()
train['Age'] = train['Age'].fillna(mean_age).round()
train['Cabin'] = train['Cabin'].notnull().astype(int)
train['Embarked'] = train['Embarked'].fillna('S').map({'S': 2, 'C': 1, 'Q': 0})
train['FamilySize'] = train['SibSp'] + train['Parch'] + 1
train['IsAlone'] = (train['FamilySize'] == 1).astype(int)
train['FareBin'] = pd.qcut(train['Fare'], 4, labels=False, duplicates='drop')
train['FareBin'] = train['FareBin'].fillna(0).astype(int)
train['Title'] = train_df['Name'].apply(extract_title)
title_encoder = LabelEncoder()
train['Title'] = title_encoder.fit_transform(train['Title'])
train['Deck'] = train_df['Cabin'].str[0].fillna('U')
deck_encoder = LabelEncoder()
train['Deck'] = deck_encoder.fit_transform(train['Deck'])

y = train['Survived'].values
X = train.drop('Survived', axis=1).values

# Test
passenger_ids = test_df['PassengerId'].copy()
test = test_df.drop(['Name', 'Ticket', 'PassengerId'], axis=1)
test['Sex'] = test['Sex'].map({'male': 0, 'female': 1})
test['Age'] = test['Age'].fillna(mean_age).round()
test['Cabin'] = test['Cabin'].notnull().astype(int)
test['Embarked'] = test['Embarked'].fillna('S').map({'S': 2, 'C': 1, 'Q': 0})
_, bins = pd.qcut(train['Fare'], 4, labels=False, retbins=True, duplicates='drop')
test['FareBin'] = pd.cut(test['Fare'], bins=bins, labels=False, include_lowest=True)
test['FareBin'] = test['FareBin'].fillna(0).astype(int)
test['FamilySize'] = test['SibSp'] + test['Parch'] + 1
test['IsAlone'] = (test['FamilySize'] == 1).astype(int)
test['Title'] = test_df['Name'].apply(extract_title)
test['Title'] = title_encoder.transform(test['Title'])
test['Deck'] = test_df['Cabin'].str[0].fillna('U')
test['Deck'] = deck_encoder.transform(test['Deck']) 

feature_cols = train.drop('Survived', axis=1).columns
test = test[feature_cols]

X_test = test.values

In [48]:
# ============== XGBoost ==============
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

xgb_params = {
    'n_estimators': 300,
    'max_depth': 3,           
    'learning_rate': 0.05,    
    'subsample': 0.8,        
    'colsample_bytree': 0.8, 
    'reg_alpha': 1.0,         
    'reg_lambda': 2.0,      
    'gamma': 0,              
    'min_child_weight': 3,   
    'objective': 'binary:logistic',
    'eval_metric': 'logloss', 
    'random_state': 42
}

xgb_model = XGBClassifier(**xgb_params, early_stopping_rounds=20)

# Training with validation
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)
eval_set = [(X_train, y_train), (X_val, y_val)]
xgb_model.fit(X_train, y_train, eval_set=eval_set, verbose=False)

# Cross-validation on a model without early_stopping
xgb_cv_model = XGBClassifier(**xgb_params)
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(xgb_cv_model, X_train, y_train, cv=skf, scoring='accuracy')
print(f'CV accuracy: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})')

CV accuracy: 0.8356 (+/- 0.0351)


In [49]:
# ============== Final predictions ==============
final_predictions = xgb_model.predict(X_test)

In [ ]:
# ============== Final csv ==============
submission = pd.DataFrame({
    'PassengerId': passenger_ids,
    'Survived': final_predictions.astype(int)
})

submission.to_csv('../../submissions/titanic/titanic-submission-v5.csv', index=False)
print(submission.head())


# Score: 0.77751

   PassengerId  Survived
0          892         0
1          893         0
2          894         0
3          895         0
4          896         1
